# Todos los indicadores del Perú en el Banco Mundial

En `03_apis_e_ia.ipynb` pedimos tres indicadores, uno por uno. Ahora la
pregunta es más ambiciosa: **¿qué sabe el Banco Mundial del Perú?**

Vamos a traer **todos** los indicadores de los *World Development Indicators*
(WDI), la base principal del Banco Mundial, para **todos los años**
disponibles. Es el mismo endpoint de siempre:

```
https://api.worldbank.org/v2/country/PER/indicator/SP.POP.TOTL
```

solo que en vez de un código vamos a pedir casi 1,500.

In [ ]:
import time

import pandas as pd
import requests

## 1. El catálogo: ¿qué indicadores existen?

El Banco Mundial tiene varias bases de datos ("fuentes"). La de los WDI es
la fuente número `2`. La API tiene un endpoint que lista todos sus
indicadores, sin datos: solo el código, el nombre y la definición.

In [ ]:
url_catalogo = "https://api.worldbank.org/v2/source/2/indicator"

respuesta = requests.get(
    url_catalogo,
    params={"format": "json", "per_page": 2000},   # 2000 para que entre todo en una página
    timeout=60,
)
respuesta.raise_for_status()

metadatos, filas_catalogo = respuesta.json()
metadatos

Cada fila del catálogo trae diccionarios anidados (`source`, `topics`).
Nos quedamos con lo que sirve para leer los datos después.

In [ ]:
catalogo = pd.DataFrame(filas_catalogo)
catalogo = catalogo[["id", "name", "sourceNote", "sourceOrganization"]]
catalogo.columns = ["codigo", "nombre", "definicion", "fuente_original"]

print("Indicadores en el catálogo:", len(catalogo))
catalogo.head()

## 2. Pedir varios indicadores en un solo pedido

Hacer 1,500 pedidos, uno por indicador, tomaría horas: cada respuesta del
Banco Mundial demora entre 20 y 40 segundos.

La API permite pedir **varios códigos a la vez, separados por `;`**,
siempre que se indique la fuente (`source=2`).

¿Cuántos caben? El límite no es la cantidad de códigos sino el **largo de la
URL**: si los códigos juntos pasan de unos 1,000 caracteres, el servidor
responde `400 Bad Request - Invalid URL`. Hay códigos largos, como
`EN.GHG.CO2.LU.DF.MT.CE.AR5`, así que usamos lotes de 40.

In [ ]:
codigos_de_prueba = ["SP.POP.TOTL", "NY.GDP.MKTP.CD", "SP.DYN.LE00.IN"]
";".join(codigos_de_prueba)

### La función que pide un lote

Es la misma idea que `pedir_indicador` del notebook anterior, con dos
cambios:

- **Reintenta.** Con tantos pedidos, alguno va a fallar por tiempo de espera.
  Si falla, espera un poco y vuelve a intentar, hasta 3 veces.
- **Revisa lo que llegó.** Esta API puede responder `200` con un error
  adentro. Si la respuesta no tiene la forma `[metadatos, datos]`, lo
  contamos como falla.

In [ ]:
def pedir_lote(codigos, pais="PER", intentos=3):
    """Trae varios indicadores del WDI para un país, todos los años."""
    url = f"https://api.worldbank.org/v2/country/{pais}/indicator/{';'.join(codigos)}"
    parametros = {"format": "json", "source": 2, "per_page": 10000}

    for intento in range(1, intentos + 1):
        try:
            r = requests.get(url, params=parametros, timeout=180)
            r.raise_for_status()
            datos = r.json()
            if len(datos) == 2 and datos[1] is not None:
                return datos[1]
            print("  Respuesta inesperada:", datos[0])
        except (requests.exceptions.RequestException, ValueError) as e:
            print(f"  Intento {intento} falló: {type(e).__name__}")
        time.sleep(5 * intento)   # cada reintento espera un poco más

    return None                   # se rindió: quien llama decide qué hacer

Primero se prueba con pocos códigos. Sin el parámetro `date`, la API
devuelve **todos los años** que tiene.

In [ ]:
filas_prueba = pedir_lote(codigos_de_prueba)
prueba = pd.DataFrame(filas_prueba)
print("Filas:", len(prueba))
prueba.head()

## 3. Recorrer todo el catálogo

Partimos la lista de códigos en lotes de 40 y los pedimos uno tras otro,
con una pausa entre pedidos para no saturar el servidor.

> ⏱️ Esta celda tarda **entre 20 y 30 minutos**: son 38 pedidos y cada
> uno demora lo que el Banco Mundial se tome en responder.

In [ ]:
todos_los_codigos = catalogo["codigo"].tolist()
tamanio_lote = 40

lotes = []
for inicio in range(0, len(todos_los_codigos), tamanio_lote):
    lotes.append(todos_los_codigos[inicio:inicio + tamanio_lote])

print("Lotes a pedir:", len(lotes))

Antes de lanzar todos los pedidos, se comprueba que ningún lote pase el
límite de caracteres. Es más barato revisarlo aquí que descubrirlo a los
15 minutos.

In [ ]:
largo_maximo = 0
for lote in lotes:
    largo_maximo = max(largo_maximo, len(";".join(lote)))

print("Caracteres del lote más largo:", largo_maximo)

In [ ]:
filas_descargadas = []
lotes_fallidos = []

for numero, lote in enumerate(lotes, start=1):
    filas = pedir_lote(lote)
    if filas is None:
        lotes_fallidos.append(lote)
    else:
        filas_descargadas.extend(filas)
    print(f"Lote {numero}/{len(lotes)}: {len(filas_descargadas):,} filas acumuladas")
    time.sleep(0.5)

print("Lotes que fallaron:", len(lotes_fallidos))

Si algún lote falló aun con los reintentos, se vuelve a pedir una vez más
al final. Normalmente esta lista queda vacía.

In [ ]:
for lote in lotes_fallidos:
    filas = pedir_lote(lote)
    if filas is not None:
        filas_descargadas.extend(filas)
        print("Recuperado un lote que había fallado")
    else:
        print("Sigue fallando, códigos:", lote)

## 4. De JSON a una tabla limpia

Igual que en el notebook anterior: el código del indicador viene anidado
dentro de `indicator`, y hay que sacarlo.

In [ ]:
crudo = pd.DataFrame(filas_descargadas)

indicadores = pd.DataFrame({
    "codigo": crudo["indicator"].apply(lambda d: d["id"]),
    "anio": crudo["date"].astype(int),
    "valor": crudo["value"],
})

print("Filas descargadas:", len(indicadores))
indicadores.head()

### La mayoría de las celdas están vacías

La API devuelve una fila por cada combinación indicador–año, **haya dato o
no**. Muchos indicadores solo se miden en algunos años, o nunca para el
Perú. Esas filas vacías se quitan.

In [ ]:
print("Filas sin valor:", indicadores["valor"].isna().sum())

indicadores = indicadores.dropna(subset=["valor"])
indicadores = indicadores.sort_values(["codigo", "anio"]).reset_index(drop=True)

print("Filas con dato:", len(indicadores))
print("Indicadores con al menos un dato:", indicadores["codigo"].nunique())
print("Años:", indicadores["anio"].min(), "a", indicadores["anio"].max())

## 5. ¿Cuánto sabemos de cada época?

Contar cuántos indicadores tienen dato cada año muestra algo importante:
la información sobre el Perú de los años 60 es mucho más pobre que la de
hoy. Y los años más recientes también tienen pocos datos, porque varios
indicadores se publican con rezago.

In [ ]:
indicadores_por_anio = indicadores.groupby("anio")["codigo"].count()
indicadores_por_anio.iloc[::5]      # uno de cada cinco años, para que se lea

## 6. Formato ancho: una fila por año

Para cruzar con otras tablas conviene tener **una fila por año y una
columna por indicador**. Es lo que hicimos con `contexto_peru.csv`, pero
ahora con todos los indicadores.

In [ ]:
indicadores_ancho = indicadores.pivot(index="anio", columns="codigo", values="valor")
print("Años x indicadores:", indicadores_ancho.shape)

indicadores_ancho[["SP.POP.TOTL", "SH.XPD.CHEX.PC.CD", "SP.RUR.TOTL.ZS"]].tail()

## 7. Guardar

Se guardan tres archivos:

- `indicadores_peru_largo.csv` — una fila por indicador y año, con el nombre.
- `indicadores_peru_ancho.csv` — una fila por año, una columna por código.
- `catalogo_indicadores_wdi.csv` — qué significa cada código.

In [ ]:
indicadores_largo = indicadores.merge(
    catalogo[["codigo", "nombre"]], on="codigo", how="left"
)
indicadores_largo = indicadores_largo[["codigo", "nombre", "anio", "valor"]]

indicadores_largo.to_csv("indicadores_peru_largo.csv", index=False)
indicadores_ancho.to_csv("indicadores_peru_ancho.csv")
catalogo.to_csv("catalogo_indicadores_wdi.csv", index=False)

indicadores_largo.shape, indicadores_ancho.shape, catalogo.shape

> 💡 **Atajo.** El Banco Mundial también ofrece todo esto en un solo archivo
> comprimido: `https://api.worldbank.org/v2/en/country/PER?downloadformat=csv`.
> Llega en un segundo, pero en un formato fijo que igual hay que limpiar. Con
> la API armamos exactamente la tabla que queremos, y la misma función sirve
> para cualquier país o cualquier lista de indicadores.

---
## Lo que hicimos

| Paso | Código |
|---|---|
| Listar los indicadores de una fuente | `requests.get(".../source/2/indicator")` |
| Pedir varios códigos a la vez | `";".join(codigos)` + `source=2` |
| Todos los años | no mandar el parámetro `date` |
| Reintentar si falla | `for intento in range(...)` + `try / except` |
| Pedir por lotes | `range(0, len(codigos), 40)` |
| Aplanar anidados | `.apply(lambda d: d["id"])` |
| Quitar filas sin dato | `.dropna(subset=["valor"])` |
| De largo a ancho | `.pivot(index="anio", columns="codigo")` |